# OAC shift analysis

Downstream anticoagulation analysis using MIE pipeline presented in 10.1016/j.jbi.2025.104898 .

<pre>
Richter-Pechanski, P., Seiferling, M., Kiriakou, C., Schwab, D. M., Geis, N. A., Dieterich, C., & Frank, A. (2025). Medication information extraction using local large language models. Journal of biomedical informatics, 104898.
</pre>

Required data:
- 2012 MIE predictions (internal data)
- 2020/21 CARDIO:DE gold annotations as UIMA CAS XMI zip files, see: https://doi.org/10.11588/DATA/AFYQDY

Workflow
1. Load 2012 model-predicted MIE output.  
2. Select medication entries whose extracted `reason` contains `koagulation`, matching the original notebook logic.  
3. Extract CARDIO:DE 2020/21 medication mentions from UIMA CAS XMI zip files using the WebAnno annotation types `webanno.custom.Medication` and `webanno.custom.MedRel`.  
4. Keep medication–reason relations where the reason annotation has `ClassType == "REASON"` and its covered text contains `koagulation`.  
5. Map medications to `VKA`, `DOAC`, or `OTHER`.  
6. Compute mention-level class shares, Wilson confidence intervals, exploratory two-proportion z-tests, and DOAC active-ingredient shares.  
7. Generate stacked bar plots to visualize OAC class shares and DOAC active-ingredient shares.ares.
es. plot.

In [ ]:
from collections import OrderedDict
from pathlib import Path
from math import sqrt, erfc
import csv
import os
import zipfile
import tempfile

import cassis
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Input files/folders
PRED_2012_CSV = Path("output_results_mie_2012.csv")
CARDIODE_CAS_DIR = Path("__folder_containing_CARDIO:DE_XMI__")

OUT_DIR = Path("outputs")
OUT_DIR.mkdir(exist_ok=True)

PREFERRED_XMI_FILES = ("admin.xmi", "INITIAL_CAS.xmi")

# CARDIO:DE WebAnno annotation types
MEDICATION_TYPE_NAME = "webanno.custom.Medication"
RELATION_TYPE_NAME = "webanno.custom.MedRel"

# Helper functions

In [ ]:
VKA_TERMS = [
    "marcumar", "macumar", "marcumer", "phenprocoumon",
    "cumarin", "marcumar-therapie", "marcumartherapie",
]

DOAC_TERMS = [
    "apixaban", "eliquis", "rivaroxaban", "xarelto",
    "edoxaban", "endoxaban", "lixiana", "lixana",
    "dabigatran", "pradaxa", "noac", "noak", "doac",
]

DOAC_MOLECULES = ["Apixaban", "Rivaroxaban", "Edoxaban", "Dabigatran"]
TARGET_REASON_SUBSTRING = "koagulation"
EXCLUDE_MEDICATION_SUBSTRING = "antikoagulation"

MEDICATION_CLASS_TYPES = {"ACTIVEING", "DRUG"}
REASON_CLASS_TYPE = "REASON"


def reason_to_text(reason) -> str:
    if isinstance(reason, list):
        return " | ".join(str(r) for r in reason if r is not None)
    return "" if reason is None else str(reason)


def is_anticoagulation_reason(reason) -> bool:
    return TARGET_REASON_SUBSTRING in reason_to_text(reason).lower()


def clean_medication_label(value: str) -> str:
    return str(value).strip().replace(" (1)", "").replace(" (2)", "")


# Keep this exact CARDIO:DE medication label although it contains
# the generic string "Antikoagulation".
ALLOWED_ANTICOAGULATION_MEDICATION_LABEL = (
    "-LRB- zusätzlich zur Antikoagulation wie u.a. -RRB-"
)


def keep_medication_label(value: str) -> bool:
    value = str(value).strip()

    if value == ALLOWED_ANTICOAGULATION_MEDICATION_LABEL:
        return True

    if not value:
        return False

    # Filter all other generic medication strings containing "antikoagulation".
    return EXCLUDE_MEDICATION_SUBSTRING not in value.lower()


def classify_oac(medication: str) -> str:
    value = (medication or "").lower()
    if any(term in value for term in VKA_TERMS):
        return "VKA"
    if any(term in value for term in DOAC_TERMS):
        return "DOAC"
    return "OTHER"


def doac_molecule(medication: str):
    value = (medication or "").lower()
    mapping = {
        "Apixaban": ["apixaban", "eliquis"],
        "Rivaroxaban": ["rivaroxaban", "xarelto"],
        "Edoxaban": ["edoxaban", "endoxaban", "lixiana", "lixana"],
        "Dabigatran": ["dabigatran", "pradaxa"],
    }
    for molecule, terms in mapping.items():
        if any(term in value for term in terms):
            return molecule
    return None


def class_type(annotation) -> str:
    return str(annotation.get("ClassType") or "")


def covered_text(annotation, cas) -> str:
    try:
        return annotation.get_covered_text()
    except Exception:
        try:
            return cas.sofa_string[annotation.begin:annotation.end]
        except Exception:
            return ""


## Load 2012 model predictions

In [ ]:
def parse_prediction(prediction: str):
    try:
        return eval(
            prediction,
            {"__builtins__": {}},
            {"OrderedDict": OrderedDict},
        )
    except Exception:
        return {}


def load_2012_predictions(path: Path) -> pd.DataFrame:
    rows = []

    with path.open("r", encoding="utf-8") as f:
        reader = csv.reader(f, delimiter="|")
        for row in reader:
            if len(row) < 2 or "pred" in row[1].lower():
                continue

            text, prediction = row[0], row[1]
            obj = parse_prediction(prediction)

            for med in obj.get("medications", []) or []:
                if not isinstance(med, dict):
                    continue

                reason = med.get("reason", "")
                if not is_anticoagulation_reason(reason):
                    continue

                name = med.get("medication", "").strip()
                if not name:
                    continue

                rows.append({
                    "year": "2012",
                    "source": path.name,
                    "text": text,
                    "medication": name,
                    "reason": reason_to_text(reason),
                })

    return pd.DataFrame(rows)

meds_2012 = load_2012_predictions(PRED_2012_CSV)
meds_2012["oac_class"] = meds_2012["medication"].apply(classify_oac)
meds_2012["doac_molecule"] = meds_2012["medication"].apply(doac_molecule)

print(f"2012 OAC-related medication mentions: {len(meds_2012)}")
display(meds_2012.head())

## Load CARDIO:DE 2020/21 XMI

In [ ]:
def open_zip(zip_path: Path):
    with zipfile.ZipFile(zip_path) as zf:
        members = zf.namelist()

        if "TypeSystem.xml" not in [Path(m).name for m in members]:
            print(f"WARNING: TypeSystem.xml not found in {zip_path.name}; skipped.")
            return None, None

        typesystem_member = next(m for m in members if Path(m).name == "TypeSystem.xml")

        xmi_member = None
        for candidate in PREFERRED_XMI_FILES:
            matches = [m for m in members if Path(m).name == candidate]
            if matches:
                xmi_member = matches[0]
                break

        if xmi_member is None:
            print(f"WARNING: no admin.xmi or INITIAL_CAS.xmi found in {zip_path.name}; skipped.")
            return None, None

        with zf.open(typesystem_member) as f:
            typesystem = cassis.load_typesystem(f)

        with zf.open(xmi_member) as f:
            cas = cassis.load_cas_from_xmi(f, typesystem=typesystem)

        return typesystem, cas


def medication_reason_pair(relation):
    """Return medication annotation and REASON annotation from a WebAnno MedRel.

    This mirrors the CARDIO:DE relation logic:
    - MedRel has Governor and Dependent features.
    - One side must be a medication annotation with ClassType ACTIVEING or DRUG.
    - The other side must be the reason annotation with ClassType REASON.
    """
    dependent = relation.get("Dependent")
    governor = relation.get("Governor")

    if dependent is None or governor is None:
        return None, None

    # Match the original CARDIO:DE handling:
    # if Dependent is the medication, switch roles so Governor is medication
    # and Dependent is the relation target, e.g. REASON.
    if class_type(dependent) in MEDICATION_CLASS_TYPES:
        dependent, governor = governor, dependent

    if class_type(governor) in MEDICATION_CLASS_TYPES and class_type(dependent) == REASON_CLASS_TYPE:
        medication_annotation = governor
        reason_annotation = dependent
        return medication_annotation, reason_annotation

    return None, None


def extract_cardiode_oac_medications(cas_dir: Path) -> pd.DataFrame:
    rows = []
    n_relations = 0
    n_reason_relations = 0
    n_koagulation_reason = 0
    n_filtered_medication_string = 0

    if not cas_dir.exists():
        raise FileNotFoundError(f"CARDIO:DE CAS directory not found: {cas_dir}")

    zip_files = sorted(cas_dir.glob("*.zip"))

    for zip_path in zip_files:
        typesystem, cas = open_zip(zip_path)

        if cas is None:
            continue

        for relation in cas.select(RELATION_TYPE_NAME):
            n_relations += 1
            medication_annotation, reason_annotation = medication_reason_pair(relation)

            if medication_annotation is None or reason_annotation is None:
                continue

            n_reason_relations += 1

            reason = covered_text(reason_annotation, cas)
            if not is_anticoagulation_reason(reason):
                continue

            n_koagulation_reason += 1

            medication = clean_medication_label(covered_text(medication_annotation, cas))
            if not keep_medication_label(medication):
                n_filtered_medication_string += 1
                continue

            rows.append({
                "year": "2020/21",
                "source": zip_path.name,
                "text": "",
                "medication": medication,
                "reason": reason,
                "relation_begin": relation.begin,
                "relation_end": relation.end,
                "medication_begin": medication_annotation.begin,
                "medication_end": medication_annotation.end,
                "reason_begin": reason_annotation.begin,
                "reason_end": reason_annotation.end,
            })

    # Do not drop duplicates: the analysis is mention-level, and repeated identical
    # medication strings in the same document must remain separate mentions.
    out = pd.DataFrame(rows).reset_index(drop=True)

    print(f"MedRel annotations inspected: {n_relations}")
    print(f"Medication-REASON relations: {n_reason_relations}")
    print(f"Relations with reason containing '{TARGET_REASON_SUBSTRING}': {n_koagulation_reason}")
    print(f"Filtered medication strings containing '{EXCLUDE_MEDICATION_SUBSTRING}': {n_filtered_medication_string}")

    if out.empty:
        raise ValueError(
            "No CARDIO:DE OAC medication mentions were extracted. "
            "Check the CAS path and the annotation types "
            f"{MEDICATION_TYPE_NAME} / {RELATION_TYPE_NAME}."
        )

    return out


meds_2020_21 = extract_cardiode_oac_medications(CARDIODE_CAS_DIR)
meds_2020_21["oac_class"] = meds_2020_21["medication"].apply(classify_oac)
meds_2020_21["doac_molecule"] = meds_2020_21["medication"].apply(doac_molecule)

meds_2020_21.to_csv(OUT_DIR / "cardiode_2020_21_oac_extracted.csv", index=False)

print(f"2020/21 OAC-related medication mentions: {len(meds_2020_21)}")
print(meds_2020_21["oac_class"].value_counts().reindex(["VKA", "DOAC", "OTHER"], fill_value=0))
display(meds_2020_21.head())

## Combine datasets and compute OAC class shares

In [ ]:
meds = pd.concat(
    [
        meds_2012[["year", "source", "text", "medication", "reason", "oac_class", "doac_molecule"]],
        meds_2020_21[["year", "source", "text", "medication", "reason", "oac_class", "doac_molecule"]],
    ],
    ignore_index=True,
)

class_counts = (
    pd.crosstab(meds["year"], meds["oac_class"])
    .reindex(index=["2012", "2020/21"], columns=["VKA", "DOAC", "OTHER"], fill_value=0)
)
class_shares = class_counts.div(class_counts.sum(axis=1), axis=0) * 100

meds.to_csv(OUT_DIR / "oac_mentions_combined.csv", index=False)
class_counts.to_csv(OUT_DIR / "oac_class_counts.csv")
class_shares.to_csv(OUT_DIR / "oac_class_shares.csv")

print("OAC class counts:")
display(class_counts)

print("OAC class shares (%):")
display(class_shares.round(1))

## Wilson confidence intervals and exploratory z-tests

In [ ]:
def wilson_ci(k: int, n: int, z: float = 1.96):
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    denom = 1 + z**2 / n
    center = p + z**2 / (2 * n)
    radius = z * sqrt((p * (1 - p) + z**2 / (4 * n)) / n)
    return ((center - radius) / denom, (center + radius) / denom)


def two_prop_ztest(k1: int, n1: int, k2: int, n2: int):
    pooled = (k1 + k2) / (n1 + n2)
    se = sqrt(pooled * (1 - pooled) * (1 / n1 + 1 / n2))
    z = (k1 / n1 - k2 / n2) / se
    p = erfc(abs(z) / sqrt(2))
    return z, p


ci_rows = []
for year in class_counts.index:
    n = int(class_counts.loc[year].sum())
    for cls in ["VKA", "DOAC", "OTHER"]:
        k = int(class_counts.loc[year, cls])
        lo, hi = wilson_ci(k, n)
        ci_rows.append({
            "year": year,
            "class": cls,
            "k": k,
            "n": n,
            "share_percent": 100 * k / n,
            "ci_low_percent": 100 * lo,
            "ci_high_percent": 100 * hi,
        })

ci_table = pd.DataFrame(ci_rows)
ci_table.to_csv(OUT_DIR / "oac_class_shares_wilson_ci.csv", index=False)

ztest_rows = []
for cls in ["DOAC", "VKA"]:
    k1 = int(class_counts.loc["2012", cls])
    n1 = int(class_counts.loc["2012"].sum())
    k2 = int(class_counts.loc["2020/21", cls])
    n2 = int(class_counts.loc["2020/21"].sum())
    z, p = two_prop_ztest(k1, n1, k2, n2)
    ztest_rows.append({
        "class": cls,
        "k_2012": k1,
        "n_2012": n1,
        "k_2020_21": k2,
        "n_2020_21": n2,
        "z": z,
        "p": p,
    })

ztests = pd.DataFrame(ztest_rows)
ztests.to_csv(OUT_DIR / "oac_two_proportion_ztests.csv", index=False)

display(ci_table.round(3))
display(ztests)

## DOAC active-ingredient composition

In [ ]:
doac = meds[meds["oac_class"] == "DOAC"].dropna(subset=["doac_molecule"])

doac_counts = (
    pd.crosstab(doac["year"], doac["doac_molecule"])
    .reindex(index=["2012", "2020/21"], columns=DOAC_MOLECULES, fill_value=0)
)
doac_shares = doac_counts.div(doac_counts.sum(axis=1), axis=0) * 100

doac_counts.to_csv(OUT_DIR / "doac_molecule_counts.csv")
doac_shares.to_csv(OUT_DIR / "doac_molecule_shares.csv")

print("DOAC active-ingredient counts:")
display(doac_counts)

print("DOAC active-ingredient shares (%):")
display(doac_shares.round(1))

# Stacked bar plots

In [ ]:
def stacked_percent(ax, table, columns, title):
    x = np.arange(len(table.index))
    bottom = np.zeros(len(table.index))

    for col in columns:
        vals = table[col].to_numpy(dtype=float)
        ax.bar(x, vals, bottom=bottom, label=col)

        for i, v in enumerate(vals):
            if v >= 6:
                ax.text(i, bottom[i] + v / 2, f"{v:.0f}%", ha="center", va="center", fontsize=8)

        bottom += vals

    ax.set_xticks(x, table.index)
    ax.set_ylim(0, 100)
    ax.set_title(title)
    ax.set_ylabel("Share of mentions (%)")
    ax.legend(frameon=False, fontsize=8)


fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.2), sharey=True)

stacked_percent(
    axes[0],
    class_shares.reindex(["2012", "2020/21"]),
    ["VKA", "DOAC", "OTHER"],
    "A  OAC classes",
)

stacked_percent(
    axes[1],
    doac_shares.reindex(["2012", "2020/21"]),
    DOAC_MOLECULES,
    "B  DOAC active ingredients",
)

fig.tight_layout()
fig.savefig(OUT_DIR / "figure2_oac_shift.png", dpi=300, bbox_inches="tight")
fig.savefig(OUT_DIR / "figure2_oac_shift.pdf", bbox_inches="tight")
plt.show()